## Dataset registration example

Its not entirely how mlflow datasets are supposed to be used, but we can leverage the codebase ( and the dataset source abstraction )
To provision flower clientapps that land in our supernode with local datasets on the fly

In [ ]:
import mlflow
import pandas as pd
import os
# Load your data
dataset_source_url = (
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv"
)
raw_data = pd.read_csv(dataset_source_url, delimiter=";")
dataset_tag = "test-tag-1" #the run/dataset name
fed_exp_id = "1" #if deployment goes well, this is always 1
# Mlflow data module method that creates the dataset object
dataset = mlflow.data.from_pandas(
    raw_data, source=dataset_source_url, name="wine-quality-white", targets="quality"
)
os.environ["MLFLOW_WORKSPACE"] = "ws-federated"
# Log the dataset to an MLflow run
# We use a convention that the run name indicates the dataset name
with mlflow.start_run(experiment_id=fed_exp_id,run_name=dataset_tag):
    #If you want to split a dataset for training<=>evaluation
    #They would have to be registered with the proper context
    #So that the clientapp eventually knows what to retrieve for which reason
    mlflow.log_input(dataset, context="training")
    mlflow.log_input(dataset, context="evaluation")

#The client app then does something like this:
_for = "training" #finally the context
client = mlflow.tracking.MlflowClient()
runs = client.search_runs(
    experiment_ids=[fed_exp_id],
    filter_string=f"attributes.run_name = '{dataset_tag}'",
    max_results=2
)
runs = [x for x in runs if x.inputs.dataset_inputs[0].tags[0].value == _for]
if len(runs) == 0:
    raise f'No {dataset_tag} dataset with context {_for}'
run = runs[0]
if len(run.inputs.dataset_inputs) == 0:
    raise 'Input run corrupt: No dataset entries'
dataset_info = run.inputs.dataset_inputs[0].dataset
dataset_source = mlflow.data.get_source(dataset_info)
#This then downloads the file on the host machine
file_path = dataset_source.load()
#In this case we know its a csv, and what the target column is
target_column = "quality"

df = pd.read_csv(file_path)
print(df)